In [1]:
import os
import gc
import math
import random
import warnings
import numpy as np
import pandas as pd
import cv2
from pathlib import Path
from tqdm.auto import tqdm
from typing import Optional

import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import LambdaLR
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.model_selection import StratifiedGroupKFold
from PIL import Image
import timm

warnings.filterwarnings('ignore')

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision("high")

print("✓ Imports complete")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"timm version: {timm.__version__}")

class CFG:
    BASE_PATH = '/kaggle/input/csiro1'
    TRAIN_CSV = os.path.join(BASE_PATH, 'train.csv')
    TRAIN_IMAGE_DIR = os.path.join(BASE_PATH, 'train')
    TEST_CSV = os.path.join(BASE_PATH, 'test.csv')
    TEST_IMAGE_DIR = os.path.join(BASE_PATH, 'test')
    
    MODEL_DIR = '/kaggle/working/models_trained'
    OUTPUT_DIR = '/kaggle/working'
    
    MODEL_NAME = 'vit_huge_plus_patch16_dinov3.lvd1689m'
    
    SEED = 42
    N_FOLDS = 3
    FOLDS_TO_TRAIN = [0, 1, 2]
    
    IMG_HEIGHT = 768
    IMG_WIDTH = 384
    BATCH_SIZE = 4
    NUM_WORKERS = 0
    
    EPOCHS = 180
    WARMUP_EPOCHS = 6
    LR_BACKBONE = 1e-5
    LR_HEAD = 5e-4
    WD = 1e-2
    
    CLIP_GRAD_NORM = 1.0
    DROPOUT = 0.2
    
    EARLY_STOPPING_PATIENCE = 30
    
    TARGETS = ["Dry_Green_g", "Dry_Dead_g", "Dry_Clover_g", "GDM_g", "Dry_Total_g"]
    TARGET_COLS = ['Dry_Green_g', 'Dry_Dead_g', 'Dry_Clover_g', 'GDM_g', 'Dry_Total_g']
    TARGET_WEIGHTS = torch.tensor([0.1, 0.1, 0.1, 0.2, 0.5])
    
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

os.makedirs(CFG.MODEL_DIR, exist_ok=True)

def seed_everything(seed=CFG.SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

seed_everything()

print(f"\n{'='*60}")
print("CONFIGURATION - VIT_HUGE_PLUS")
print(f"{'='*60}")
print(f"Device: {CFG.DEVICE}")
print(f"Model: {CFG.MODEL_NAME}")
print(f"Image Size: {CFG.IMG_HEIGHT}x{CFG.IMG_WIDTH}")
print(f"Batch Size: {CFG.BATCH_SIZE}")
print(f"Epochs: {CFG.EPOCHS}")
print(f"Folds: {CFG.N_FOLDS}")

print(f"\n{'='*60}")
print("STEP 1: Loading Data")
print(f"{'='*60}")

def load_train_data():
    df = pd.read_csv(CFG.TRAIN_CSV)
    df['image_id'] = df['sample_id'].str.split('__').str[0]
    
    df_wide = df.pivot_table(
        index=['image_id', 'image_path'],
        columns='target_name',
        values='target',
        aggfunc='first'
    ).reset_index()
    
    for col in CFG.TARGET_COLS:
        if col not in df_wide.columns:
            df_wide[col] = 0.0
    
    df_wide['total_bin'] = pd.qcut(df_wide['Dry_Total_g'], q=5, labels=False, duplicates='drop')
    
    sgkf = StratifiedGroupKFold(n_splits=CFG.N_FOLDS, shuffle=True, random_state=CFG.SEED)
    df_wide['fold'] = -1
    
    for fold, (_, val_idx) in enumerate(sgkf.split(df_wide, df_wide['total_bin'], groups=df_wide['image_id'])):
        df_wide.loc[val_idx, 'fold'] = fold
    
    print(f"✓ Loaded {len(df_wide)} training images")
    print(f"Fold distribution:\n{df_wide['fold'].value_counts().sort_index()}")
    return df_wide

def load_test_data():
    df = pd.read_csv(CFG.TEST_CSV)
    df['image_id'] = df['sample_id'].str.split('__').str[0]
    df_unique = df.drop_duplicates('image_id')[['image_id', 'image_path']].reset_index(drop=True)
    print(f"✓ Loaded {len(df_unique)} test images")
    return df_unique

train_df = load_train_data()
test_df = load_test_data()

def get_train_transforms():
    return A.Compose([
        A.Resize(CFG.IMG_HEIGHT, CFG.IMG_WIDTH),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.1, rotate_limit=15, p=0.5),
        A.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.05, p=0.3),
        A.Normalize(mean=[0.4417, 0.5036, 0.3057], std=[0.2364,0.2355,0.2219]),
        ToTensorV2()
    ])

def get_val_transforms():
    return A.Compose([
        A.Resize(CFG.IMG_HEIGHT, CFG.IMG_WIDTH),
        A.Normalize(mean=[0.4417, 0.5036, 0.3057], std=[0.2364,0.2355,0.2219]),
        ToTensorV2()
    ])

class BiomassDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform
        self.paths = df['image_path'].values
        self.labels = df[CFG.TARGET_COLS].values.astype(np.float32)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_name = os.path.basename(self.paths[idx])
        path = os.path.join(self.img_dir, img_name)
        
        img = cv2.imread(path)
        if img is None:
            img = np.zeros((1000, 2000, 3), dtype=np.uint8)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        h, w, _ = img.shape
        mid = w // 2
        left = img[:, :mid]
        right = img[:, mid:]
        
        if self.transform:
            left = self.transform(image=left)['image']
            right = self.transform(image=right)['image']
        
        label = torch.from_numpy(self.labels[idx])
        return left, right, label

print("✓ Dataset class defined")

class LocalMambaBlock(nn.Module):
    def __init__(self, dim: int, kernel_size: int = 5, dropout: float = 0.1):
        super().__init__()
        self.norm = nn.LayerNorm(dim)
        self.dwconv = nn.Conv1d(dim, dim, kernel_size=kernel_size, padding=kernel_size // 2, groups=dim)
        self.gate = nn.Linear(dim, dim)
        self.proj = nn.Linear(dim, dim)
        self.drop = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        shortcut = x
        x = self.norm(x)
        g = torch.sigmoid(self.gate(x))
        x = x * g
        x = x.transpose(1, 2)
        x = self.dwconv(x)
        x = x.transpose(1, 2)
        x = self.proj(x)
        x = self.drop(x)
        return shortcut + x

class BiomassModel(nn.Module):
    def __init__(self, model_name: str, pretrained: bool = True):
        super().__init__()
        self.model_name = model_name
        
        self.backbone = timm.create_model(
            model_name, 
            pretrained=pretrained, 
            num_classes=0, 
            global_pool=''
        )
        nf = self.backbone.num_features
        print(f"✓ Backbone: {model_name}, features={nf}")
        
        self.fusion = nn.Sequential(
            LocalMambaBlock(nf, kernel_size=5, dropout=CFG.DROPOUT),
            LocalMambaBlock(nf, kernel_size=5, dropout=CFG.DROPOUT)
        )
        self.pool = nn.AdaptiveAvgPool1d(1)
        
        self.head_green = nn.Sequential(
            nn.Linear(nf, nf // 2), nn.GELU(), nn.Dropout(CFG.DROPOUT),
            nn.Linear(nf // 2, 1), nn.Softplus()
        )
        self.head_dead = nn.Sequential(
            nn.Linear(nf, nf // 2), nn.GELU(), nn.Dropout(CFG.DROPOUT),
            nn.Linear(nf // 2, 1), nn.Softplus()
        )
        self.head_clover = nn.Sequential(
            nn.Linear(nf, nf // 2), nn.GELU(), nn.Dropout(CFG.DROPOUT),
            nn.Linear(nf // 2, 1), nn.Softplus()
        )

    def forward(self, left, right):
        x_l = self.backbone(left)
        x_r = self.backbone(right)
        x_cat = torch.cat([x_l, x_r], dim=1)
        x_fused = self.fusion(x_cat)
        x_pool = self.pool(x_fused.transpose(1, 2)).flatten(1)
        
        green = self.head_green(x_pool)
        dead = self.head_dead(x_pool)
        clover = self.head_clover(x_pool)
        gdm = green + clover
        total = gdm + dead
        
        return torch.cat([green, dead, clover, gdm, total], dim=1)

print("✓ Model architecture defined")

class WeightedBiomassLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.weights = CFG.TARGET_WEIGHTS
        self.huber = nn.SmoothL1Loss(reduction='none', beta=5.0)
    
    def forward(self, preds, labels):
        preds_log = torch.log1p(preds)
        labels_log = torch.log1p(labels)
        
        loss = self.huber(preds_log, labels_log)
        weighted_loss = (loss * self.weights.to(loss.device)).mean()
        return weighted_loss

def weighted_r2_score(y_true, y_pred):
    weights = CFG.TARGET_WEIGHTS.numpy()
    r2_scores = []
    
    y_true_log = np.log1p(y_true)
    y_pred_log = np.log1p(y_pred)
    
    for i in range(y_true.shape[1]):
        yt = y_true_log[:, i]
        yp = y_pred_log[:, i]
        ss_res = np.sum((yt - yp) ** 2)
        ss_tot = np.sum((yt - np.mean(yt)) ** 2)
        r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 0.0
        r2_scores.append(r2)
    
    r2_scores = np.array(r2_scores)
    weighted = np.sum(r2_scores * weights) / np.sum(weights)
    return weighted, r2_scores

print("✓ Loss and metrics defined")

def build_optimizer(model):
    backbone_params = list(model.backbone.parameters())
    backbone_ids = {id(p) for p in backbone_params}
    head_params = [p for p in model.parameters() if id(p) not in backbone_ids]
    
    return optim.AdamW([
        {'params': backbone_params, 'lr': CFG.LR_BACKBONE},
        {'params': head_params, 'lr': CFG.LR_HEAD}
    ], weight_decay=CFG.WD)

def build_scheduler(optimizer, total_steps):
    def lr_lambda(step):
        warmup_steps = CFG.WARMUP_EPOCHS * (total_steps // CFG.EPOCHS)
        if step < warmup_steps:
            return float(step) / float(max(1, warmup_steps))
        progress = (step - warmup_steps) / float(max(1, total_steps - warmup_steps))
        return 0.5 * (1.0 + math.cos(math.pi * progress))
    return LambdaLR(optimizer, lr_lambda)

scaler = GradScaler()

def train_one_epoch(model, loader, optimizer, scheduler, criterion, device):
    model.train()
    total_loss = 0.0
    
    pbar = tqdm(loader, desc='Training')
    for i, (left, right, labels) in enumerate(pbar):
        left = left.to(device)
        right = right.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        
        with autocast():
            preds = model(left, right)
            loss = criterion(preds, labels)
        
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.CLIP_GRAD_NORM)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        
        total_loss += loss.item()
        pbar.set_postfix({'loss': f'{total_loss/(i+1):.4f}'})
    
    return total_loss / len(loader)

@torch.no_grad()
def validate(model, loader, device):
    model.eval()
    all_preds = []
    all_labels = []
    
    for left, right, labels in tqdm(loader, desc='Validating'):
        left = left.to(device)
        right = right.to(device)
        
        with autocast():
            preds = model(left, right)
        
        all_preds.append(preds.cpu().numpy())
        all_labels.append(labels.numpy())
    
    all_preds = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)
    
    weighted_r2, per_target_r2 = weighted_r2_score(all_labels, all_preds)
    return weighted_r2, per_target_r2

print("✓ Training functions defined")

print(f"\n{'='*60}")
print("STEP 2: Training DINO HUGE Models")
print(f"{'='*60}")

def train_fold(fold, train_df):
    print(f"\n{'='*60}")
    print(f"TRAINING FOLD {fold}")
    print(f"{'='*60}")
    
    train_data = train_df[train_df['fold'] != fold].reset_index(drop=True)
    val_data = train_df[train_df['fold'] == fold].reset_index(drop=True)
    
    print(f"Train: {len(train_data)}, Val: {len(val_data)}")
    
    train_dataset = BiomassDataset(train_data, CFG.TRAIN_IMAGE_DIR, get_train_transforms())
    val_dataset = BiomassDataset(val_data, CFG.TRAIN_IMAGE_DIR, get_val_transforms())
    
    train_loader = DataLoader(
        train_dataset, 
        batch_size=CFG.BATCH_SIZE, 
        shuffle=True, 
        num_workers=CFG.NUM_WORKERS, 
        pin_memory=True, 
        drop_last=True
    )
    val_loader = DataLoader(
        val_dataset, 
        batch_size=CFG.BATCH_SIZE, 
        shuffle=False, 
        num_workers=CFG.NUM_WORKERS, 
        pin_memory=True
    )
    
    model = BiomassModel(CFG.MODEL_NAME, pretrained=True).to(CFG.DEVICE)
    criterion = WeightedBiomassLoss()
    
    optimizer = build_optimizer(model)
    total_steps = len(train_loader) * CFG.EPOCHS
    scheduler = build_scheduler(optimizer, total_steps)
    
    best_r2 = -float('inf')
    best_epoch = 0
    epochs_without_improvement = 0
    
    epoch_pbar = tqdm(range(CFG.EPOCHS), desc=f'Fold {fold} Epochs')
    for epoch in epoch_pbar:
        print(f"\nEpoch {epoch + 1}/{CFG.EPOCHS}")
        
        train_loss = train_one_epoch(model, train_loader, optimizer, scheduler, criterion, CFG.DEVICE)
        val_r2, per_r2 = validate(model, val_loader, CFG.DEVICE)
        
        print(f"Train Loss: {train_loss:.4f}")
        print(f"Val R²: {val_r2:.4f}")
        print(f"Per-target: Green={per_r2[0]:.3f}, Dead={per_r2[1]:.3f}, Clover={per_r2[2]:.3f}, GDM={per_r2[3]:.3f}, Total={per_r2[4]:.3f}")
        
        epoch_pbar.set_postfix({'loss': f'{train_loss:.4f}', 'val_r2': f'{val_r2:.4f}', 'best_r2': f'{best_r2:.4f}'})
        
        if val_r2 > best_r2:
            best_r2 = val_r2
            best_epoch = epoch + 1
            epochs_without_improvement = 0
            save_path = f"{CFG.MODEL_DIR}/fold{fold}_best.pth"
            torch.save(model.state_dict(), save_path)
            print(f"✓ Saved best model (R²={best_r2:.4f})")
        else:
            epochs_without_improvement += 1
            print(f"No improvement for {epochs_without_improvement} epoch(s)")
            
            if epochs_without_improvement >= CFG.EARLY_STOPPING_PATIENCE:
                print(f"Early stopping triggered after {epoch + 1} epochs")
                break
    
    print(f"\nFold {fold} Best: R²={best_r2:.4f} at epoch {best_epoch}")
    
    del model, optimizer, scheduler, train_loader, val_loader, criterion
    gc.collect()
    torch.cuda.empty_cache()
    
    return best_r2

fold_scores = []
fold_pbar = tqdm(CFG.FOLDS_TO_TRAIN, desc='Training Folds')
for fold in fold_pbar:
    fold_pbar.set_description(f'Training Fold {fold}')
    score = train_fold(fold, train_df)
    fold_scores.append(score)
    fold_pbar.set_postfix({'current_r2': f'{score:.4f}', 'mean_r2': f'{np.mean(fold_scores):.4f}'})

print(f"\n{'='*60}")
print("DINO HUGE TRAINING COMPLETE!")
print(f"{'='*60}")
print(f"Fold scores: {fold_scores}")
print(f"Mean CV R²: {np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f}")

summary = {
    'model': CFG.MODEL_NAME,
    'folds': CFG.N_FOLDS,
    'epochs': CFG.EPOCHS,
    'batch_size': CFG.BATCH_SIZE,
    'image_size': f'{CFG.IMG_HEIGHT}x{CFG.IMG_WIDTH}',
    'fold_scores': [float(score) for score in fold_scores],  # Convert np.float32 to Python float
    'mean_cv': float(np.mean(fold_scores)),  # Convert np.float64 to Python float
    'std_cv': float(np.std(fold_scores))  # Convert np.float64 to Python float
}

import json
with open(f'{CFG.OUTPUT_DIR}/training_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print(f"\n{'='*60}")
print("TRAINING COMPLETE!")
print(f"{'='*60}")
print(f"\nModels saved to: {CFG.MODEL_DIR}/")
print(f"  - fold0_best.pth")
print(f"  - fold1_best.pth")
print(f"  - fold2_best.pth")
print(f"\nNext steps:")
print(f"1. Create Kaggle dataset from /kaggle/working/")
print(f"2. Use inference notebook to submit")

/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

✓ Imports complete
PyTorch: 2.8.0+cu126
CUDA: True
GPU: NVIDIA H100 80GB HBM3
VRAM: 85.3 GB
timm version: 1.0.20

CONFIGURATION - VIT_HUGE_PLUS
Device: cuda
Model: vit_huge_plus_patch16_dinov3.lvd1689m
Image Size: 768x384
Batch Size: 4
Epochs: 180
Folds: 3

STEP 1: Loading Data
✓ Loaded 357 training images
Fold distribution:
fold
0    119
1    120
2    118
Name: count, dtype: int64
✓ Loaded 1 test images
✓ Dataset class defined
✓ Model architecture defined
✓ Loss and metrics defined
✓ Training functions defined

STEP 2: Training DINO HUGE Models


Training Folds:   0%|          | 0/3 [00:00<?, ?it/s]


TRAINING FOLD 0
Train: 238, Val: 119


model.safetensors:   0%|          | 0.00/3.36G [00:00<?, ?B/s]

...

Validating:   0%|          | 0/30 [00:00<?, ?it/s]

Train Loss: 0.0002
Val R²: 0.8188
Per-target: Green=0.713, Dead=0.764, Clover=0.820, GDM=0.842, Total=0.841
No improvement for 29 epoch(s)

Epoch 70/180


Training:   0%|          | 0/59 [00:00<?, ?it/s]

Validating:   0%|          | 0/30 [00:00<?, ?it/s]

Train Loss: 0.0002
Val R²: 0.8148
Per-target: Green=0.735, Dead=0.778, Clover=0.806, GDM=0.831, Total=0.833
No improvement for 30 epoch(s)
Early stopping triggered after 70 epochs

Fold 2 Best: R²=0.8222 at epoch 40

DINO HUGE TRAINING COMPLETE!
Fold scores: [np.float32(0.8377088), np.float32(0.87147355), np.float32(0.8222146)]
Mean CV R²: 0.8438 ± 0.0206

TRAINING COMPLETE!

Models saved to: /kaggle/working/models_trained/
  - fold0_best.pth
  - fold1_best.pth
  - fold2_best.pth

Next steps:
1. Create Kaggle dataset from /kaggle/working/
2. Use inference notebook to submit
